In [9]:
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")

CSV_PATH = "Walmart.csv"
MIN_ROWS_PER_STORE = 20
p, d, q = 1, 0, 1  # fallback order

def thanksgiving_date(year):
    d = pd.Timestamp(year=year, month=11, day=1)
    first_thursday = 1 + ((3 - d.weekday()) % 7)
    return pd.Timestamp(year=year, month=11, day=first_thursday + 21).date()

# ...existing code...
def add_features(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    # ensure ordering per store for lag/rolling/ewm creation
    df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

    years = df['Date'].dt.year.dropna().unique()
    tg = {int(y): thanksgiving_date(int(y)) for y in years}

    def is_major(d):
        if pd.isna(d): return 0
        if (d.month == 12 and d.day == 25): return 1
        return 1 if tg.get(int(d.year)) == d.date() else 0

    df['is_major_holiday'] = df['Date'].apply(is_major).astype(int)
    if 'Holiday_Flag' in df.columns:
        df['is_minor_holiday'] = ((df['Holiday_Flag'] == 1) & (df['is_major_holiday'] == 0)).astype(int)
    else:
        df['is_minor_holiday'] = 0

    # time features
    df['year'] = df['Date'].dt.year.astype('Int64')
    df['month'] = df['Date'].dt.month.astype('Int64')
    try:
        df['weekofyear'] = df['Date'].dt.isocalendar().week.astype('Int64')
    except Exception:
        df['weekofyear'] = df['Date'].dt.weekofyear.astype('Int64')
    df['day_of_week'] = df['Date'].dt.weekday.astype('Int64')

    # cyclical encodings
    df['weekofyear_sin'] = np.sin(2 * np.pi * (df['weekofyear'].astype(float) - 1) / 52)
    df['weekofyear_cos'] = np.cos(2 * np.pi * (df['weekofyear'].astype(float) - 1) / 52)
    df['month_sin'] = np.sin(2 * np.pi * (df['month'].astype(float) - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month'].astype(float) - 1) / 12)

    # ensure Weekly_Sales numeric for feature creation
    df['Weekly_Sales'] = pd.to_numeric(df.get('Weekly_Sales'), errors='coerce')

    # --- LAG features per store (from Weekly_Sales) ---
    lag_periods = [1, 2, 3, 7, 14, 28, 52]
    for lag in lag_periods:
        df[f'lag_{lag}'] = df.groupby('Store')['Weekly_Sales'].shift(lag)

    # --- Rolling window statistics per store ---
    roll_means = [3, 4, 8, 12, 52]
    roll_stds = [3, 4, 8, 52]
    for w in roll_means:
        df[f'roll_mean_{w}'] = df.groupby('Store')['Weekly_Sales'].transform(
            lambda x: x.rolling(window=w, min_periods=1).mean()
        )
    for w in roll_stds:
        df[f'roll_std_{w}'] = df.groupby('Store')['Weekly_Sales'].transform(
            lambda x: x.rolling(window=w, min_periods=1).std()
        )

    # --- EWMA features per store ---
    ewm_spans = [3, 7, 14]
    for s in ewm_spans:
        df[f'ewm_span_{s}'] = df.groupby('Store')['Weekly_Sales'].transform(
            lambda x: x.ewm(span=s, adjust=False).mean()
        )

     # --- Transformed / normalized target ---
    # safe log(1 + sales)
    df['sales_log1p'] = np.log1p(df['Weekly_Sales'].fillna(0))

    return df

def safe_numeric(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = 0
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
    # fill small gaps
    df[cols] = df[cols].fillna(method='ffill').fillna(method='bfill').fillna(0)
    return df

def rmse_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def main():
    df = pd.read_csv(CSV_PATH)
    df = add_features(df)

# ...existing code...
    # include lag + rolling + ewm + store-stats features in exogenous set
    exog_features = [
        'is_major_holiday','is_minor_holiday',
        'Temperature','Fuel_Price','CPI','Unemployment',
        'month_sin','month_cos','weekofyear_sin','weekofyear_cos',
        'year','month','weekofyear','day_of_week',
        # lag features
        'lag_1','lag_2','lag_3','lag_7','lag_14','lag_28','lag_52',
        # rolling means
        'roll_mean_3','roll_mean_4','roll_mean_8','roll_mean_12','roll_mean_52',
        # rolling stds
        'roll_std_3','roll_std_4','roll_std_8','roll_std_52',
        # EWMA features
        'ewm_span_3','ewm_span_7','ewm_span_14'
    ]
# ...existing code...
    # convert and fill numeric columns (this will fill NaNs from lag/rolling creation)
    df = safe_numeric(df, exog_features + ['Weekly_Sales'])
    df = df.sort_values(['Store','Date']).reset_index(drop=True)

    # ...existing code continues unchanged...

    stores = sorted(df['Store'].dropna().unique())
    mae_base_list = []; mae_exog_list = []
    rmse_base_list = []; rmse_exog_list = []
    n_test_list = []

    for idx, store_id in enumerate(stores, 1):
        sd = df[df['Store'] == store_id].copy().sort_values('Date')
        sd = sd.dropna(subset=['Date'])
        sd['Weekly_Sales'] = pd.to_numeric(sd['Weekly_Sales'], errors='coerce')
        series = sd.set_index('Date')['Weekly_Sales'].dropna().sort_index()
        if series.empty:
            print(f"Store {store_id}: skipped (no sales after dropna)")
            continue
        # align exog to series index and ensure numeric
        exog = sd.set_index('Date')[exog_features].reindex(series.index)
        exog = exog.apply(pd.to_numeric, errors='coerce').fillna(method='ffill').fillna(method='bfill').fillna(0)

        n = len(series)
        if n < MIN_ROWS_PER_STORE:
            print(f"Store {store_id}: skipped (n={n} < {MIN_ROWS_PER_STORE})")
            continue
        train_end = int(0.7 * n); val_end = int(0.85 * n)
        if train_end < 3 or (n - val_end) < 1:
            print(f"Store {store_id}: skipped due to small splits (n={n})")
            continue

        train = series.iloc[:train_end].astype(float)
        val = series.iloc[train_end:val_end].astype(float)
        test = series.iloc[val_end:].astype(float)
        train_exog = exog.iloc[:train_end].astype(float)
        val_exog = exog.iloc[train_end:val_end].astype(float)
        test_exog = exog.iloc[val_end:].astype(float)

        if train.std() == 0 or train.nunique() == 1:
            print(f"Store {store_id}: skipped (train constant)")
            continue

        try:
            # Baseline
            base_mod = SARIMAX(train.values, order=(p,d,q))
            base_res = base_mod.fit(disp=False)
            train_val = pd.concat([train, val]) if len(val) > 0 else train
            base_tv = SARIMAX(train_val.values, order=(p,d,q)).fit(disp=False)
            if len(test) > 0:
                fc_base = base_tv.get_forecast(steps=len(test))
                pred_base = np.asarray(fc_base.predicted_mean, dtype=float)
                mae_b = float(mean_absolute_error(test.values, pred_base))
                rmse_b = rmse_np(test.values, pred_base)
            else:
                mae_b = np.nan; rmse_b = np.nan

            # With exog
            ex_mod = SARIMAX(train.values, exog=train_exog.values, order=(p,d,q))
            ex_res = ex_mod.fit(disp=False)
            train_val_exog = np.vstack([train_exog.values, val_exog.values]) if len(val_exog) > 0 else train_exog.values
            train_val_series = np.concatenate([train.values, val.values]) if len(val) > 0 else train.values
            ex_tv = SARIMAX(train_val_series, exog=train_val_exog, order=(p,d,q)).fit(disp=False)
            if len(test) > 0:
                fc_ex = ex_tv.get_forecast(steps=len(test), exog=test_exog.values)
                pred_ex = np.asarray(fc_ex.predicted_mean, dtype=float)
                mae_e = float(mean_absolute_error(test.values, pred_ex))
                rmse_e = rmse_np(test.values, pred_ex)
            else:
                mae_e = np.nan; rmse_e = np.nan

            mae_base_list.append(mae_b); mae_exog_list.append(mae_e)
            rmse_base_list.append(rmse_b); rmse_exog_list.append(rmse_e)
            n_test_list.append(len(test))

            delta_mae = mae_b - mae_e if not (np.isnan(mae_b) or np.isnan(mae_e)) else np.nan
            delta_rmse = rmse_b - rmse_e if not (np.isnan(rmse_b) or np.isnan(rmse_e)) else np.nan

            print(f"Store {store_id} ({idx}/{len(stores)}) - test_rows={len(test)} | "
                  f"BASE MAE={mae_b:.2f} RMSE={rmse_b:.2f} | EXOG MAE={mae_e:.2f} RMSE={rmse_e:.2f} | "
                  f"MAE_delta={delta_mae:.2f} RMSE_delta={delta_rmse:.2f}")

        except Exception as e:
            print(f"Store {store_id}: error during modeling -> {e}")
            continue

    # Summary
    if len(mae_base_list) == 0:
        print("No stores processed successfully.")
        return

    mae_base = np.array(mae_base_list, dtype=float)
    mae_exog = np.array(mae_exog_list, dtype=float)
    rmse_base = np.array(rmse_base_list, dtype=float)
    rmse_exog = np.array(rmse_exog_list, dtype=float)
    n_test = np.array(n_test_list, dtype=float)

    simple_mae_base = np.nanmean(mae_base)
    simple_mae_exog = np.nanmean(mae_exog)
    weighted_mae_base = np.nansum(mae_base * n_test) / np.nansum(n_test)
    weighted_mae_exog = np.nansum(mae_exog * n_test) / np.nansum(n_test)

    simple_rmse_base = np.nanmean(rmse_base)
    simple_rmse_exog = np.nanmean(rmse_exog)
    weighted_rmse_base = np.nansum(rmse_base * n_test) / np.nansum(n_test)
    weighted_rmse_exog = np.nansum(rmse_exog * n_test) / np.nansum(n_test)

    print("\nSUMMARY ACROSS STORES")
    print(f"Stores evaluated: {len(mae_base_list)}")
    print(f"Simple MAE - Baseline: {simple_mae_base:.2f}, Exog: {simple_mae_exog:.2f}, Delta: {simple_mae_base - simple_mae_exog:.2f}")
    print(f"Weighted MAE - Baseline: {weighted_mae_base:.2f}, Exog: {weighted_mae_exog:.2f}, Delta: {weighted_mae_base - weighted_mae_exog:.2f}")
    print(f"Simple RMSE - Baseline: {simple_rmse_base:.2f}, Exog: {simple_rmse_exog:.2f}, Delta: {simple_rmse_base - simple_rmse_exog:.2f}")
    print(f"Weighted RMSE - Baseline: {weighted_rmse_base:.2f}, Exog: {weighted_rmse_exog:.2f}, Delta: {weighted_rmse_base - weighted_rmse_exog:.2f}")

if __name__ == "__main__":
    main()

Store 1 (1/45) - test_rows=22 | BASE MAE=79356.00 RMSE=93869.30 | EXOG MAE=20.83 RMSE=29.24 | MAE_delta=79335.17 RMSE_delta=93840.06
Store 2 (2/45) - test_rows=22 | BASE MAE=59275.57 RMSE=72108.43 | EXOG MAE=11050.94 RMSE=13179.34 | MAE_delta=48224.63 RMSE_delta=58929.10
Store 3 (3/45) - test_rows=22 | BASE MAE=11788.45 RMSE=15523.45 | EXOG MAE=3902.79 RMSE=4459.77 | MAE_delta=7885.66 RMSE_delta=11063.68
Store 4 (4/45) - test_rows=22 | BASE MAE=49459.61 RMSE=60565.36 | EXOG MAE=8821.04 RMSE=10635.46 | MAE_delta=40638.57 RMSE_delta=49929.90
Store 5 (5/45) - test_rows=22 | BASE MAE=17327.33 RMSE=19602.61 | EXOG MAE=8508.27 RMSE=10191.43 | MAE_delta=8819.06 RMSE_delta=9411.18
Store 6 (6/45) - test_rows=22 | BASE MAE=84774.90 RMSE=109237.67 | EXOG MAE=12778.34 RMSE=14857.67 | MAE_delta=71996.56 RMSE_delta=94380.00
Store 7 (7/45) - test_rows=22 | BASE MAE=142995.22 RMSE=159208.07 | EXOG MAE=38332.10 RMSE=42939.15 | MAE_delta=104663.12 RMSE_delta=116268.92
Store 8 (8/45) - test_rows=22 | BAS

Loading data from Walmart.csv...
Feature Engineering: Starting...
Feature Engineering: Complete.
Saving to features.csv...
Success! features.csv created.
